# Save, Load, and Export Fit Results

This notebook is the canonical reference for the `FitResults` archive API — the HDF5 round-trip (`save_fit` / `FitResults.load`) and its one-way CSV/PNG sibling (`export_fit`). See [`10_model_comparison`](../10_model_comparison/example.ipynb) for the fits' details; this notebook focuses on what to do with the results after a comparison.

**The two channels**

- **`save_fit` → HDF5.** Structured, lossless, σ-snapshot included. Round-trips back into trspecfit via `FitResults.load`. For *future you* and other trspecfit users.
- **`export_fit` → CSV + PNG tree.** One-way (no `load` counterpart). For Origin, MATLAB, paper plots, non-trspecfit colleagues.

**Preamble (next cell)** — `%run`s notebook 10 in the kernel so all of its variables (`file`, `project`, the fitted `baseA/baseB/sbsA/sbsB/m2dA/m2dB` slots) are in scope below. The mechanics:

1. `%cd` into notebook 10's directory so its `project.yaml` and model YAMLs resolve.
2. `%%capture` wraps the whole cell so the noisy run (notebook 10 keeps `show_output: 1` for its own interactive use) doesn't dump into this notebook's output. `auto_export: false` in notebook 10's `project.yaml` keeps the run from writing CSV/PNG side effects either.
3. `%cd -` back so any HDF5/CSV/PNG artifacts this notebook writes land in `11_save_load_export/`.

Expected runtime: ~30–40 s. The cell after the preamble prints a one-line confirmation so you know the fits landed.

In [ ]:
%%capture
# %cd into notebook 10's dir so its project.yaml + model YAMLs resolve;
# %run executes that notebook in the current kernel (variables stay live);
# %cd - returns to this notebook's dir so save_fit / export_fit write here.
# %%capture swallows notebook 10's noisy output without changing its config —
# its show_output: 1 stays correct for direct interactive use.
# The path is quoted: %%capture tokenizes the cell body as Python, and a
# bare ../10_model_comparison reads as a malformed number ("10_model...").
%cd -q "../10_model_comparison"
%run example.ipynb
%cd -q -

In [ ]:
# Heartbeat: confirm notebook 10's pipeline landed in this kernel.
print(f"file:   {file.name}")
print(f"models: {project.results.models(file=file.name)}")
print(f"slots:  {len(project._fit_history)} in _fit_history")

## 1. Save → Load → Compare Across Sessions

`file.save_fit(path)` is sugar for `project.save_fits(path, file=file)`. By default it writes a snapshot of `_fit_history` (latest slot per canonical key; `keep_history=True` is deferred to v2). `FitResults.load(path)` reads the archive without needing a `Project`, so this is the persistence path that survives kernel restarts and travels well between machines.

Each saved slot carries its own σ snapshot (`noise_type`, `sigma_source`, `sigma_type`, `sigma_data`, `sigma_eff`), so a loaded archive shows the **same calibrated columns** the live session did — no need to re-`set_sigma` on the recipient side.

**What if you disagree with the saved σ?** The raw column `chi2_red_raw` is always present, so a what-if recalibration is one line of pandas — no rehydrated Project required:

```python
df = loaded.compare_models(fit_type="2d")  # single-file archive: file= optional
df["chi2_red_alt"] = df["chi2_red_raw"] / alt_sigma**2
```

In [ ]:
archive_path = "comparison.fit.h5"
file.save_fit(archive_path, overwrite=True)

loaded = trspecfit.FitResults.load(archive_path)
# Pull the file id from the archive itself — not the live `file` from the
# preamble — so the read cells below also run in a fresh "load only" session.
archive_file = loaded.files()[0]
print(loaded)

In [ ]:
# Same comparison API, but driven from the on-disk archive — no Project required.
# Each slot stores its σ snapshot at fit time, so calibrated columns appear here
# even though we never called set_sigma() on this fresh FitResults.
loaded.compare_models(file=archive_file, fit_type="baseline")

In [ ]:
loaded.compare_models(
    file=archive_file, fit_type="sbs", sbs_aggregation="long"
).head(6)

## 2. Ship Just the Winning Fits

The verdict from notebook 10: `baseA` wins at baseline, `m2dA` wins at 2D. Once you've read it, the natural next step is to ship those specific slots — leaving losers, SbS exploration, and the rest of the history behind.

The same `model=` / `fit_type=` filters that drive `compare_models` also drive `save_fit` and `export_fit`, so picking one slot is a one-liner. The two channels frame the audience choice:

- **`save_fit` → single `.fit.h5`.** Lossless, σ-snapshot included, round-trips back via `FitResults.load`. For another trspecfit user or future you.
- **`export_fit` → CSV + PNG tree.** One-way. For a 2D fit you get `params.csv`, `metrics.csv`, `fit_2d.csv`, `observed_2d.csv`, `energy.csv`, `time.csv`, plus `2D_data_fit_res.png`; baseline yields a slimmer `fit_1d.csv` (energy / observed / fit / residual) alongside the same params/metrics.

In [ ]:
# Save just the winning baseline and 2D slots to standalone h5 archives —
# the same model= / fit_type= filters used in compare_models pick one slot.
file.save_fit("winner_base.fit.h5", model="baseA", fit_type="baseline", overwrite=True)
file.save_fit("winner_2d.fit.h5",   model="m2dA", fit_type="2d",       overwrite=True)

# Confirm each archive holds exactly one slot.
print(trspecfit.FitResults.load("winner_base.fit.h5"))
print(trspecfit.FitResults.load("winner_2d.fit.h5"))

In [ ]:
# Same filters steer the CSV/PNG export. Each call writes a directory rooted
# at the given path; baseline yields a slim 1D tree, 2D yields the full
# heatmap PNG plus observed/fit/energy/time CSVs.
file.export_fit("winner_base", model="baseA", fit_type="baseline", overwrite=True)
file.export_fit("winner_2d",   model="m2dA", fit_type="2d",       overwrite=True)

# Show what landed on disk for both exports — the contrast between channels.
from pathlib import Path
for root in ("winner_base", "winner_2d"):
    print(f"--- {root}/")
    for p in sorted(Path(root).rglob("*")):
        if p.is_file():
            print(f"  {p}")

## 3. Browse the Loaded Archive

Beyond `compare_models`, `FitResults` exposes a small query API: `files()`, `models()`, `find()` (multi-match), and `get()` (exactly one match, raises otherwise). Slots carry their fingerprint, identity, params DataFrame, observed/fit arrays, and metrics — everything needed to inspect a fit without rehydrating the model graph.

In [ ]:
print("files:  ", loaded.files())
print("models: ", loaded.models())

slot = loaded.get(file=archive_file, model="baseA", fit_type="baseline")
print(f"\nbaseA baseline slot:")
print(f"  observed shape: {slot.observed.shape}")
print(f"  fit shape:      {slot.fit.shape}")
print(f"  metrics:        {slot.metrics}")
print(f"  params rows:    {len(slot.params)}")
slot.params

### Anatomy of a slot

`get()` / `find()` return a `SavedFitSlot` — a frozen dataclass. The cell above pulled four of its fields; introspecting it lists every field you can extract — with array/frame shapes and dict keys, so you can see what each one holds — including the `selection` dict, the σ snapshot (`sigma_data` / `sigma_eff`), confidence intervals (`conf_ci`), and the MCMC chain (`mcmc`) when present. Every field is documented in the `SavedFitSlot` docstring — there's no need to open the `.h5` to discover what's inside.

In [ ]:
import dataclasses

for f in dataclasses.fields(slot):
    value = getattr(slot, f.name)
    if hasattr(value, "shape"):
        detail = f"{type(value).__name__}{value.shape}"
    elif isinstance(value, dict):
        detail = f"dict[{', '.join(value.keys())}]"
    elif value is None:
        detail = "None"
    else:
        detail = type(value).__name__
    print(f"{f.name:18} {detail}")

## Tips

- **Default path:** `file.save_fit()` / `project.save_fits()` write to `./fit_results/<project_name>.fit.h5` when no path is given; `file.export_fit()` / `project.export_fits()` write to `./fit_results/<project_name>/`. Pass an explicit path to override (as the cells above do).
- **Slot-scoped overwrite:** re-saving with the same `(file, model, fit_type, selection)` raises `FileExistsError` unless `overwrite=True`. Append-by-default: writing to an existing archive augments it.
- **`auto_export` side effect:** `fit_*` methods auto-write CSV/PNG into `project.path_results` on completion by default. Notebook 10's `project.yaml` sets `auto_export: false` to keep that quiet; toggle it (or call `project.auto_export = False`) in any session where you only want explicit export calls to write to disk.
- **`File.set_sigma` is the only sigma entry point.** Calibrated columns (`chi2`, `chi2_red`) and stored metrics (`aic`, `bic`, …) survive load without re-`set_sigma` — they were materialized at fit time and live inside each slot's σ snapshot. For what-if recalibration of *existing* slots, use the always-present `chi2_red_raw` column (see preamble §1).

**Next steps:**
- Uncertainty estimation — CI tables and MCMC chains ride along in saved slots (`conf_ci`, `mcmc`): [`12_uncertainty_mcmc`](../12_uncertainty_mcmc/example.ipynb)
- The same archive workflow across many files at once: [`20_fit_each_separately`](../20_fit_each_separately/example.ipynb)